In [1]:
from pathlib import Path
import pandas as pd
import re
import torch
import torch.nn.functional as F


In [2]:
# 1. 下载 VidProM 原始 prompt

data_path = Path("VidProM_unique.csv")
if not data_path.is_file():
    raise FileNotFoundError(f"Dataset not found: {data_path.resolve()}")

df = pd.read_csv(
    data_path,
    usecols=["uuid", "prompt"],
)

print(df.shape)
print(df.columns.tolist())
df.head()

(1672243, 2)
['uuid', 'prompt']


,uuid,prompt
0,6a83eb92-faa0-572b-9e1f-67dec99b711d,"Flying among clouds and stars, kitten Max disc..."
1,3ba1adf3-5254-59fb-a13e-57e6aa161626,Use a clean and modern font for the text &quot...
2,f88d8fff-1c26-59d2-8252-23ee67762b12,dancing sticks .screen size must be 9:16
3,871c822c-2a73-5fa6-b4ff-67f821813be1,promt robots hugging
4,3cbb0e22-a3b4-5841-a5a1-12db24ae656a,"ridley scott space documentary, 2020"


In [3]:
# 2. 长度过滤并固定为 50,000 条
SEED = 42

df = pd.read_csv(
    "VidProM_unique.csv",
    usecols=["uuid", "prompt"],
)

df = df.dropna(subset=["uuid", "prompt"]).copy()

df["prompt"] = (
    df["prompt"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df = df[
    (df["prompt"] != "")
    & (~df["uuid"].duplicated())
].copy()

df["word_count"] = df["prompt"].str.count(
    r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*"
)

candidates = df[
    df["word_count"].between(5, 30)
].copy()

if len(candidates) < 50_000:
    raise RuntimeError(
        f"过滤后只有 {len(candidates)} 条，需要放宽长度范围"
    )

filtered = candidates.sample(
    n=50_000,
    random_state=SEED,
).reset_index(drop=True)

filtered.to_csv("chorus_filtered_50000.csv", index=False)

print("长度过滤候选数:", len(candidates))
print("最终数量:", len(filtered))

长度过滤候选数: 1189999
最终数量: 50000


In [ ]:
# 3. 生成 768 维 CLIP embedding

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from tqdm import tqdm
from transformers import AutoTokenizer, CLIPTextModelWithProjection

MODEL_NAME = "openai/clip-vit-large-patch14"
BATCH_SIZE = 256

df = pd.read_csv("chorus_filtered_50000.csv")

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = CLIPTextModelWithProjection.from_pretrained(
    MODEL_NAME
).eval().to(device)

if device == "cuda":
    model = model.half()

embedding_batches = []

with torch.inference_mode():
    for start in tqdm(range(0, len(df), BATCH_SIZE)):
        prompts = df["prompt"].iloc[
            start:start + BATCH_SIZE
        ].tolist()

        inputs = tokenizer(
            prompts,
            padding=True,
            truncation=True,
            max_length=77,
            return_tensors="pt",
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        embeddings = model(**inputs).text_embeds
        embeddings = F.normalize(
            embeddings.float(),
            p=2,
            dim=1,
        )

        embedding_batches.append(
            embeddings.cpu().numpy()
        )

embeddings = np.concatenate(embedding_batches, axis=0)

assert embeddings.shape == (50_000, 768)
assert np.allclose(
    np.linalg.norm(embeddings, axis=1),
    1.0,
    atol=1e-5,
)

np.save("chorus_embeddings_50000.npy", embeddings)

print(embeddings.shape)

/opt/homebrew/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 19709.89it/s]
[transformers] CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...23}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.pre_

(50000, 768)


In [ ]:
# 4. 对归一化 embedding 做 KMeans

import joblib
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans

SEED = 42

df = pd.read_csv("chorus_filtered_50000.csv")
embeddings = np.load("chorus_embeddings_50000.npy")

kmeans = KMeans(
    n_clusters=20,
    random_state=SEED,
    n_init=10,
    algorithm="lloyd",
    verbose=1,
)

labels = kmeans.fit_predict(embeddings)

df["cluster_id"] = labels
df.to_csv("chorus_clustered_50000.csv", index=False)

np.save("chorus_cluster_labels.npy", labels)
joblib.dump(kmeans, "chorus_kmeans.joblib")

cluster_sizes = np.bincount(labels, minlength=20)

for cluster_id, size in enumerate(cluster_sizes):
    print(cluster_id, size)

Initialization complete
Iteration 0, inertia 49016.1328125.
Iteration 1, inertia 30402.48828125.
Iteration 2, inertia 30044.75.
Iteration 3, inertia 29919.65234375.
Iteration 4, inertia 29839.419921875.
Iteration 5, inertia 29790.478515625.
Iteration 6, inertia 29754.27734375.
Iteration 7, inertia 29729.361328125.
Iteration 8, inertia 29711.693359375.
Iteration 9, inertia 29698.4140625.
Iteration 10, inertia 29687.962890625.
Iteration 11, inertia 29679.4375.
Iteration 12, inertia 29672.01953125.
Iteration 13, inertia 29664.33203125.
Iteration 14, inertia 29656.525390625.
Iteration 15, inertia 29647.482421875.
Iteration 16, inertia 29638.41796875.
Iteration 17, inertia 29629.4921875.
Iteration 18, inertia 29619.177734375.
Iteration 19, inertia 29606.095703125.
Iteration 20, inertia 29592.171875.
Iteration 21, inertia 29586.181640625.
Iteration 22, inertia 29583.271484375.
Iteration 23, inertia 29581.173828125.
Iteration 24, inertia 29579.462890625.
Iteration 25, inertia 29577.875.
Itera

In [4]:
import joblib
import numpy as np
import pandas as pd

df = pd.read_csv("chorus_clustered_50000.csv")
embeddings = np.load("chorus_embeddings_50000.npy")
kmeans = joblib.load("chorus_kmeans.joblib")

labels = df["cluster_id"].to_numpy()

print("各 cluster 大小：")
print(
    df.groupby("cluster_id")
    .size()
    .rename("size")
    .to_string()
)

representatives = []

for cluster_id in range(20):
    indices = np.flatnonzero(labels == cluster_id)
    center = kmeans.cluster_centers_[cluster_id]

    # 找距离该 cluster 中心最近的 prompt
    distances = np.linalg.norm(
        embeddings[indices] - center,
        axis=1,
    )

    top_indices = indices[
        np.argsort(distances)[:10]
    ]

    print(f"\n===== Cluster {cluster_id}，共 {len(indices)} 条 =====")

    for rank, row_index in enumerate(top_indices, start=1):
        prompt = df.iloc[row_index]["prompt"]
        uuid = df.iloc[row_index]["uuid"]

        print(f"{rank}. {prompt}")

        representatives.append({
            "cluster_id": cluster_id,
            "cluster_size": len(indices),
            "rank": rank,
            "uuid": uuid,
            "prompt": prompt,
            "distance_to_center": float(
                np.linalg.norm(
                    embeddings[row_index] - center
                )
            ),
        })

pd.DataFrame(representatives).to_csv(
    "cluster_representatives.csv",
    index=False,
)

各 cluster 大小：
cluster_id
0     2968
1      823
2     2170
3     3500
4     3227
5     1750
6      794
7     4250
8     4235
9     1587
10    1445
11    2414
12    1624
13     739
14    2547
15    7565
16    2878
17    1797
18    2505
19    1182

===== Cluster 0，共 2968 条 =====
1. a man in a suit facing the camera, talking in the middle of a war scene, apocalyps, cinematic scene, anime style, realistic
2. A boy riding a horse on a green meadow
3. A Person sitting in the room and playing black colour guitar.
4. young man in an office working on the computer looking at camera
5. a man in a rabbit fur coat walking towards the camer with explosions all around him
6. dark haired man in a suit playing guitar
7. A man on a white horse with a long sword with sun in the background
8. A small child studying at a desk with a desk lamp
9. Inside the spaceship, a boy is seen floating, looking out of the window at the stars
10. Jesus wearing a white robe dancing in the streets of a modern city with ma

In [5]:
from pathlib import Path

import pandas as pd


INPUT_CSV = "chorus_clustered_50000.csv"
OUTPUT_DIR = Path("clusters")

df = pd.read_csv(INPUT_CSV)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for cluster_id in range(20):
    cluster_df = df.loc[
        df["cluster_id"] == cluster_id,
        ["uuid", "prompt"],
    ].reset_index(drop=True)

    output_file = (
        OUTPUT_DIR
        / f"cluster_{cluster_id:02d}_{len(cluster_df)}.csv"
    )

    cluster_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig",
    )

    print(f"Cluster {cluster_id}: {len(cluster_df)} 条 → {output_file}")

Cluster 0: 2968 条 → clusters/cluster_00_2968.csv
Cluster 1: 823 条 → clusters/cluster_01_823.csv
Cluster 2: 2170 条 → clusters/cluster_02_2170.csv
Cluster 3: 3500 条 → clusters/cluster_03_3500.csv
Cluster 4: 3227 条 → clusters/cluster_04_3227.csv
Cluster 5: 1750 条 → clusters/cluster_05_1750.csv
Cluster 6: 794 条 → clusters/cluster_06_794.csv
Cluster 7: 4250 条 → clusters/cluster_07_4250.csv
Cluster 8: 4235 条 → clusters/cluster_08_4235.csv
Cluster 9: 1587 条 → clusters/cluster_09_1587.csv
Cluster 10: 1445 条 → clusters/cluster_10_1445.csv
Cluster 11: 2414 条 → clusters/cluster_11_2414.csv
Cluster 12: 1624 条 → clusters/cluster_12_1624.csv
Cluster 13: 739 条 → clusters/cluster_13_739.csv
Cluster 14: 2547 条 → clusters/cluster_14_2547.csv
Cluster 15: 7565 条 → clusters/cluster_15_7565.csv
Cluster 16: 2878 条 → clusters/cluster_16_2878.csv
Cluster 17: 1797 条 → clusters/cluster_17_1797.csv
Cluster 18: 2505 条 → clusters/cluster_18_2505.csv
Cluster 19: 1182 条 → clusters/cluster_19_1182.csv


In [ ]:
# 5. 选择约 2,000 条的一个簇

import numpy as np
import pandas as pd

SEED = 42

df = pd.read_csv("chorus_clustered_50000.csv")
embeddings = np.load("chorus_embeddings_50000.npy")
labels = np.load("chorus_cluster_labels.npy")

cluster_sizes = np.bincount(labels, minlength=20)

eligible_clusters = np.flatnonzero(
    cluster_sizes >= 2_000
)

if len(eligible_clusters) == 0:
    raise RuntimeError("没有包含至少 2000 条的簇")

cluster_id = int(
    eligible_clusters[
        np.argmin(
            np.abs(
                cluster_sizes[eligible_clusters] - 2_000
            )
        )
    ]
)

member_indices = np.flatnonzero(
    labels == cluster_id
)

rng = np.random.default_rng(SEED)

selected_indices = rng.choice(
    member_indices,
    size=2_000,
    replace=False,
)

workload = df.iloc[selected_indices].copy()
workload["filtered_row"] = selected_indices
workload = workload.reset_index(drop=True)

workload_embeddings = embeddings[selected_indices]

workload.to_csv(
    "chorus_workload_2000.csv",
    index=False,
)

np.save(
    "chorus_workload_embeddings_2000.npy",
    workload_embeddings,
)

print("选择的 cluster:", cluster_id)
print("原 cluster 大小:", cluster_sizes[cluster_id])
print("最终 workload:", len(workload))